In [0]:
import os , sys

current = os.getcwd()
reusable = os.path.join(current , ".." , "..")
print(reusable)
sys.path.append(reusable)
from utils.modules import *


In [0]:
%fs ls "abfss://bronze@spotifyproject.dfs.core.windows.net/DimTrack/"

In [0]:
df_track = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimTrack/checkpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@spotifyproject.dfs.core.windows.net/DimTrack/*")

In [0]:
obj = Transform()

In [0]:

df_track = obj.dropColumn(df_track,['_rescued_data'])
df_track = (df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")
.when(col('duration_sec')<300,"medium")
.otherwise("high")))

df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))



In [0]:
df_track.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@spotifyproject.dfs.core.windows.net/DimTrack/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@spotifyproject.dfs.core.windows.net/DimTrack/data")\
            .toTable("spotify_catalog.silver.DimTrack")

In [0]:
%sql
select * from spotify_catalog.silver.dimtrack